# UC3 - RAG Product & Vendor Q&A (GlobalMart Gold)

## 1) Install (run once per cluster)

In [0]:
%pip install sentence-transformers faiss-cpu openai

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import json, faiss, subprocess, sys, uuid
from datetime import datetime, timezone
from typing import Dict, List, Tuple
import numpy as np
import pandas as pd
from openai import OpenAI
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType, TimestampType
from sentence_transformers import SentenceTransformer


/local_disk0/.ephemeral_nfs/envs/pythonEnv-d021a154-1ac1-4530-9cfa-2474ae0ea3fd/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


## 2) Configuration, and UC table

In [0]:
CATALOG = "globalmart.gold"
RAG_HISTORY_TABLE = f"{CATALOG}.rag_query_history"
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL_NAME = "databricks-gpt-oss-20b"
TOP_K = 12

## 3) Create query history table


In [0]:
spark.sql(
    f"""
CREATE TABLE IF NOT EXISTS {RAG_HISTORY_TABLE} (
  query_id STRING COMMENT 'Unique ID',
  question STRING COMMENT 'User question asked to the RAG system',
  answer STRING COMMENT 'Grounded answer produced by the LLM',
  retrieved_documents STRING COMMENT 'JSON array of retrieved doc metadata + text',
  asked_at TIMESTAMP COMMENT 'UTC timestamp when question was answered'
) USING DELTA
COMMENT 'RAG query logs for product/vendor Q&A';
"""
)

DataFrame[]

## 4) Resolve source tables and load data

In [0]:
def resolve_first_existing_table(candidates: List[str]) -> str:
    for name in candidates:
        if spark.catalog.tableExists(name):
            return name
    raise ValueError(f"None of these tables exist: {candidates}")


PRODUCT_TABLE = resolve_first_existing_table([f"{CATALOG}.dim_products", f"{CATALOG}.dim_product"])
VENDOR_TABLE = resolve_first_existing_table([f"{CATALOG}.dim_vendors", f"{CATALOG}.dim_vendor"])
PRODUCT_REGION_MV = resolve_first_existing_table([f"{CATALOG}.agg_product_region_monthly"])
VENDOR_RETURN_MV = resolve_first_existing_table([f"{CATALOG}.agg_vendor_return_rate"])

products_df = spark.table(PRODUCT_TABLE)
vendors_df = spark.table(VENDOR_TABLE)
product_mv_df = spark.table(PRODUCT_REGION_MV)
vendor_mv_df = spark.table(VENDOR_RETURN_MV)

# display(products_df.limit(5))
# display(vendors_df.limit(5))

## 5) Enrich dimensions with agg views (simple join)


In [0]:
products_enriched = products_df.join(product_mv_df, on="product_id", how="left")
vendors_enriched = vendors_df.join(vendor_mv_df, on="vendor_id", how="left")

category_candidates = [c for c in ["categories", "category", "sub_category"] if c in products_enriched.columns]
if category_candidates:
    products_enriched = products_enriched.withColumn("categories_raw", F.coalesce(*[F.col(c) for c in category_candidates]))
else:
    products_enriched = products_enriched.withColumn("categories_raw", F.lit(None).cast("string"))

# display(products_enriched.limit(5))
# display(vendors_enriched.limit(5))

## 6) Convert rows to natural-language documents

In [0]:
def row_to_product_doc(row: Dict) -> Dict:
    product_id = row.get("product_id")
    product_name = row.get("product_name")
    region_name = row.get("region_name", "unknown")
    year_month = row.get("year_month")
    categories = row.get("categories_raw") or "unknown"

    doc_text = (
        f"Product {product_name} (product_id: {product_id}) in region {region_name}. "
        f"Categories: {categories}. "
        f"Metrics: total_sales={row.get('total_sales')}, total_quantity={row.get('total_quantity')}, "
        f"total_profit={row.get('total_profit')}, order_count={row.get('order_count')}, year_month={year_month}. "
        f"Use this for product performance and regional availability analysis."
    )
    return {
        "doc_type": "product",
        "doc_id": f"product::{product_id}::{region_name}::{year_month}",
        "entity_key": str(product_id),
        "region_name": str(region_name),
        "text": doc_text,
    }


def row_to_vendor_doc(row: Dict) -> Dict:
    vendor_id = row.get("vendor_id")
    vendor_name = row.get("vendor_name")
    doc_text = (
        f"Vendor {vendor_name} (vendor_id: {vendor_id}). "
        f"Metrics: vendor_gross_sales={row.get('vendor_gross_sales')}, "
        f"vendor_refunds={row.get('vendor_refunds')}, refund_to_sales_ratio={row.get('refund_to_sales_ratio')}. "
        f"Use this for vendor return-rate comparison."
    )
    return {
        "doc_type": "vendor",
        "doc_id": f"vendor::{vendor_id}",
        "entity_key": str(vendor_id),
        "region_name": None,
        "text": doc_text,
    }


product_rows = [r.asDict(recursive=True) for r in products_enriched.collect()]
vendor_rows = [r.asDict(recursive=True) for r in vendors_enriched.collect()]
documents: List[Dict] = [row_to_product_doc(r) for r in product_rows] + [row_to_vendor_doc(r) for r in vendor_rows]

print(f"Prepared {len(documents)} documents.")
# display(pd.DataFrame(documents).head(10))

Prepared 8816 documents.


## 7) Local embeddings + FAISS index

In [0]:
embedder = SentenceTransformer(EMBED_MODEL_NAME)
doc_texts = [d["text"] for d in documents]
doc_embeddings = embedder.encode(doc_texts, convert_to_numpy=True, normalize_embeddings=True)

index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings.astype(np.float32))
print(f"FAISS index ready with {index.ntotal} vectors, dim={doc_embeddings.shape[1]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS index ready with 8816 vectors, dim=384


## 8) Databricks OpenAI-compatible client

In [0]:
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
_ws_raw = spark.conf.get("spark.databricks.workspaceUrl")
_ws_url = _ws_raw.replace("https://", "").replace("http://", "").strip("/")

client = OpenAI(api_key=DATABRICKS_TOKEN, base_url=f"https://{_ws_url}/serving-endpoints")

## 9) Retrieval + grounded answering

In [0]:
SYSTEM_PROMPT = """You are GlobalMart's retrieval QA assistant.
Answer ONLY using the provided retrieved documents.
Do not use markdown tables, bullet lists, or headings
If the answer is not supported by retrieved documents, reply exactly:
"I do not have enough information in the retrieved documents to answer this question."
Write concise business English and complete sentences."""


def extract_text_from_llm_content(raw_content) -> str:
    if raw_content is None:
        return ""
    if isinstance(raw_content, str):
        return raw_content.strip()
    if isinstance(raw_content, list):
        out = []
        for item in raw_content:
            if isinstance(item, str) and item.strip():
                out.append(item.strip())
            elif isinstance(item, dict):
                txt = item.get("text") or item.get("content")
                if isinstance(txt, str) and txt.strip():
                    out.append(txt.strip())
        return "\n\n".join(out).strip()
    if isinstance(raw_content, dict):
        txt = raw_content.get("text") or raw_content.get("content")
        if isinstance(txt, str):
            return txt.strip()
    return str(raw_content).strip()


def retrieve_docs(question: str, top_k: int = TOP_K) -> List[Dict]:
    q_emb = embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    scores, idxs = index.search(q_emb, top_k)
    hits = []
    for score, i in zip(scores[0], idxs[0]):
        if i < 0:
            continue
        d = documents[int(i)].copy()
        d["score"] = float(score)
        hits.append(d)
    return hits


def answer_question(question: str, top_k: int = TOP_K) -> Tuple[str, List[Dict]]:
    hits = retrieve_docs(question, top_k=top_k)
    context = "\n\n".join([h["text"] for h in hits])
    user_prompt = f"Question:\n{question}\n\nRetrieved documents:\n{context}\n\nAnswer from retrieved docs only."

    resp = client.chat.completions.create(
        model=LLM_MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT}, 
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.0,
        max_tokens=700,
    )
    raw = resp.choices[0].message.content if resp.choices and resp.choices[0].message else ""
    answer = extract_text_from_llm_content(raw)
    if not answer:
        answer = "I do not have enough information in the retrieved documents to answer this question."
    return answer, hits


def log_query(question: str, answer: str, hits: List[Dict]) -> None:
    query_id = str(uuid.uuid4())
    payload = json.dumps(hits, ensure_ascii=True)
    schema = StructType(
        [
            StructField("query_id", StringType(), False),
            StructField("question", StringType(), False),
            StructField("answer", StringType(), False),
            StructField("retrieved_documents", StringType(), False),
            StructField("asked_at", TimestampType(), False),
        ]
    )
    out = spark.createDataFrame([(query_id, question, answer, payload, datetime.now(timezone.utc))], schema=schema)
    out.write.mode("append").saveAsTable(RAG_HISTORY_TABLE)

def ask_and_log(question: str, top_k: int = TOP_K) -> Dict:
    answer, hits = answer_question(question, top_k=top_k)
    query_id = log_query(question, answer, hits)
    return {"query_id": query_id, "question": question, "answer": answer, "retrieved_docs": hits}


## 10) User-driven Q&A (recommended)

In [0]:
RUN_DEMO = True

if RUN_DEMO:
    test_questions = [
        "Which products are slow-moving in the West region?",
        "Which vendor has the highest return rate?",
        "What are the top-selling categories in the East region?",
        "Which products appear in East but have very low quantity sold?",
        "Do we have evidence for product availability in the Central region for women shoes?",
]
    for q in test_questions:
        out = ask_and_log(q)
        print(f"\nQ: {q}\nA: {out['answer']}\n")



Q: Which products are slow-moving in the West region?
A: The West region shows that the Soft Flip‑flops (product_id OFF‑BI‑10004654) and the Frontier Natural Mantra Legging Pant (product_id FUR‑FU‑10004963) have comparatively low sales and quantities, indicating they are slow‑moving products in that region.


Q: Which vendor has the highest return rate?
A: The vendor with the highest return rate is Johnsons Logistics (vendor_id VEN04).


Q: What are the top-selling categories in the East region?
A: The East region’s highest‑selling categories are Clothing and Shoes, each with combined sales of about $165.20.


Q: Which products appear in East but have very low quantity sold?
A: The only product listed for the East region is the Frontier Natural Products Co‑op Mantra Legging Pant44 Krazy Karma44 Large (product_id: FUR‑FU‑10004963), which sold 8 units.


Q: Do we have evidence for product availability in the Central region for women shoes?
A: Yes, the retrieved documents show that women